# Notebook contendo o treinamento e teste de um modelo de ViT

Aqui consta o passo a passo para que você consiga importar um modelo de ViT, realizar um ajuste do modelo, testá-lo e aplicá-lo em um conjunto de imagens próprio.

Neste caso, usaremos o framework do pytorch para realizar essas etapas. É necessário que você tenha o seu próprio dataset montado, de preferência salvo no drive para que seja usado via google colab

# 1 - Importação:

Primeiro é necessário importar um conjunto de bibliotecas que serão usadas para o treinamento do modelo.

In [ ]:
# Caso vá usar no drive e fazer uso de algum reositório guardado aí, monte o drive e use o caminho do drive para acessar os arquivos. Exemplo:
# from google.colab import drive
import matplotlib.pyplot as plt
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.models import vit_b_16, ViT_B_16_Weights


Este trecho de código utiliza como dispositivo uma placa de vídeo caso esteja disponível, em caso contrário utiliza cpu

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# 2 - Importação dos dados

Em nosso caso, nossa base de dados está salva no Drive e será importada de lá para ser utilizada.

In [ ]:
# drive.mount('/content/drive')

In [ ]:
# Armazena o caminho principal e separa em teste, treino e validação

ds_path ="ADICIONE AQUI SEU CAMINHO/"
train_path = os.path.join(ds_path, "train")
val_path = os.path.join(ds_path, "val")
test_path = os.path.join(ds_path, "test")

# 3 - Importação do modelo e pesos

Importe os pesos da própria biblioteca do PyTorch. É importante utilizar também os métodos de preprocessamento do modelo para que seja possível realizar o treinamento.

In [ ]:
weights = ViT_B_16_Weights.DEFAULT
preprocess = weights.transforms()

In [ ]:
train_dataset = ImageFolder(train_path, transform=preprocess)
val_dataset = ImageFolder(val_path, transform=preprocess)
test_dataset = ImageFolder(test_path, transform=preprocess)

In [ ]:
batch_size = 32
num_workers = 2 if torch.cuda.is_available() else 0
pin_memory = True if torch.cuda.is_available() else False

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

In [ ]:
model = vit_b_16(weights=weights)
in_features = model.heads.head.in_features
model.heads.head = nn.Linear(in_features, 2)
model = model.to(device)

# 4 - Configuração dos hiperparâmetros

Configure os hiperparâmetros que melhor se adequem ao seu tipo de tarefa. Sinta-se livre para testar diversas combinações até encontrar uma que se adeque melhor ao seu tipo de classificação.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = StepLR(optimizer, step_size=7, gamma=0.1)

# 5 - Definir função para o treinamento

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
  model.train()
  running_loss = 0.0
  running_correct = 0
  total = 0

  # Já separa em images e targets no loader
  for images, targets in loader:

    images, targets = images.to(device), targets.to(device)

    # Zera o gradiente
    optimizer.zero_grad()

    # Realliza a previsão para a imagem específica
    outputs = model(images)

    # Calcula a perda e o gradiente baseado na função determinada anteriormente
    loss = criterion(outputs, targets)
    loss.backward()

    # Ajusta os pesos
    optimizer.step()

    # Computa os dados para cálculo futuro
    running_loss += loss.item() * images.size(0)
    preds = outputs.argmax(dim=1)
    running_correct += (preds == targets).sum().item()
    total += images.size(0)

  # Apresenta os dados calculados
  epoch_loss = running_loss / total
  epoch_acc = running_correct / total
  return epoch_loss, epoch_acc

In [ ]:
# Desliga cálculo de gradiente e avalia o modelo
@torch.no_grad()
def evaluate(model, loader, criterion, device):
  model.eval()
  running_loss = 0.0
  running_correct = 0
  total = 0

  for images, targets in loader:
    images, targets = images.to(device), targets.to(device)
    outputs = model(images)
    loss = criterion(outputs, targets)

    running_loss += loss.item() * images.size(0)
    preds = outputs.argmax(dim=1)
    running_correct += (preds == targets).sum().item()
    total += images.size(0)

  epoch_loss = running_loss / total
  epoch_acc = running_correct / total
  return epoch_loss, epoch_acc

# 6 - Treinamento

Realiza o treinamento do modelo e reescreve a época com o melhor resultado

In [ ]:
best_val_acc = 0.0
best_ckpt_path = "vit_b16_best.pt"
num_epochs = 5

# Realiza o treinamento por época
for epoch in range(num_epochs):
  print(f"Dei inicio na epoca {epoch}")
  train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
  val_loss, val_acc = evaluate(model, val_loader, criterion, device)
  scheduler.step()

  print(f"Epoch {epoch+1}/{num_epochs} | "
          f"train_loss={train_loss:.4f} acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} acc={val_acc:.4f}")

  if val_acc > best_val_acc:
      best_val_acc = val_acc
      torch.save({
          "model_state": model.state_dict(),
          "optimizer_state": optimizer.state_dict(),
          "epoch": epoch + 1,
          "val_acc": best_val_acc,
          "weights_enum": str(weights)
      }, best_ckpt_path)
      print(f"Novo melhor checkpoint => {best_ckpt_path} (val_acc={best_val_acc:.4f})")

# 7 - Carregar o melhor checkpoint de treinamento

Carrega o melhor checkpoint dado seu caminho e realiza a avaliação do modelo naquele checkpoint.

In [ ]:
best_ckpt_path = "vit_b16_best.pt"

if os.path.exists(best_ckpt_path):
    ckpt = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    print(f"Checkpoint carregado (época {ckpt['epoch']}, val_acc={ckpt['val_acc']:.4f})")

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Teste | loss={test_loss:.4f} acc={test_acc:.4f}")

# 8 - Monta matriz de confusão e relatório de classificação para estatísticas

Note que nesse caso temos um grande desbalanceamento para a classe 0. Nosso conjunto apresenta muito mais imagens de classe 0 do que de classe 1.

In [ ]:
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

# Pressupõe: device, model, preprocess (weights.transforms()), test_path já definidos e modelo carregado em eval().

y_true, y_pred = [], []

# Classe 0
for fname in os.listdir(os.path.join(test_path, '0')):
    img_path = os.path.join(test_path, '0', fname)
    try:
        img = Image.open(img_path).convert("RGB")
    except:
        continue
    tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)         # [1, 2]
        probs = F.softmax(logits, 1)
    pred = probs.argmax(1).item()
    y_true.append(0)
    y_pred.append(pred)

# Classe 1
for fname in os.listdir(os.path.join(test_path, '1')):
    img_path = os.path.join(test_path, '1', fname)
    try:
        img = Image.open(img_path).convert("RGB")
    except:
        continue
    tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)
        probs = F.softmax(logits, 1)
    pred = probs.argmax(1).item()
    y_true.append(1)
    y_pred.append(pred)

# Matriz de confusão
labels = [0, 1]
cm = confusion_matrix(y_true, y_pred, labels=labels)  # cm: [[TN, FP], [FN, TP]]
print("Matriz de confusão (contagens):")
print(cm)

# Visualização (opcional)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[str(l) for l in labels])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix - Test")
plt.show()

# Relatório de classificação
report = classification_report(y_true, y_pred, target_names=['0','1'], digits=4)
print("Relatório de Classificação:")
print(report)


In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
import numpy as np

model.eval()
all_true, all_pred = [], []

with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        targets = targets.to(device)

        logits = model(images)
        preds = torch.argmax(F.softmax(logits, 1), dim=1)
        all_true.extend(targets.cpu().tolist())
        all_pred.extend(preds.cpu().tolist())

labels = [0, 1]

cm = confusion_matrix(all_true, all_pred, labels=labels)
print("Matriz de confusão (contagens):")
print(cm)

cm_norm = cm.astype(np.float64)
cm_norm = cm_norm / cm_norm.sum(axis=1, keepdims=True)
cm_percent = cm_norm * 100.0

print("Matriz de confusão (% por linha):")
print(cm_percent)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm,
                              display_labels=[str(l) for l in labels])
disp.plot(cmap="Blues", values_format=".1%")
plt.title("Confusion Matrix - Test (normalized)")
plt.show()



# 9 - Uso do modelo para imagens em contextos reais

Agora utilizo outras ferramentas para importar imagens de um contexto real e classificá-las utilizando o modelo. As imagens são fornecidas via link em uma tabela, então importamos a tabela, verificamos linha por linha os dados necessários e verificamos a validade do link. Após isso transformamos a imagem de acordo com a necessidade do modelo e classificamos ela sem utilizar o cálculo de gradiente.

In [ ]:
import pandas as pd
import os
from io import BytesIO
import requests

path_de_teste = "ADICIONE AQUI SEU CAMINHO/test.csv"

df = pd.read_csv(path_de_teste)
df = df.drop(["COLUNAS INDESEJADAS"], axis=1)
df

In [ ]:
def verifica_link(url):
  try:
    response = requests.get(url)
    if response.status_code == 404 or response.status_code == 403:
      return False
    else:
      return BytesIO(response.content)
  except:
    return False

In [ ]:
data_dict = {}

for _, row in df.iterrows():
  url = verifica_link(row["URL"])
  if url:
    img = Image.open(url).convert("RGB")
    data_dict[row["index"]] = img

data_dict.keys()

In [ ]:
data_dict[4]

In [ ]:
previsoes = {}

for index,img in data_dict.items():
  tensor = preprocess(img).unsqueeze(0).to(device)

  with torch.no_grad():
    logits = model(tensor)
    probs = F.softmax(logits, dim=1)

  _,preds = probs.max(dim=1)
  pred_class = preds.item()
  previsoes[index] = pred_class

previsoes